# 11 — CatBoost bootstrap comparison

Compare Bayesian bootstrap with the saved depth-6 MVS control. Run one configuration at a time. Keep features, splits, early stopping, and inner-OOF threshold selection unchanged. Depth 6 is a provisional development choice, not an independently validated optimum.

In [1]:
import json
import sys
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from IPython.display import display
from sklearn.metrics import cohen_kappa_score, confusion_matrix, f1_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split

_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from src.config import (
    ID_COLUMN, TARGET, TRAIN_PATH, TEST_PATH, RESULTS_DIR, RANDOM_STATE,
)
from src.evaluation import create_cv_splits
from src.features import build_features

## 1. Configuration

Start with `BAGGING_TEMPERATURE = 1.0`. The reference is `results/catboost_depth/depth_6`; it is not modified. `sampling_frequency="PerTree"` matches the saved MVS models. Restart the kernel and run all cells after any configuration change. Rerunning the same configuration replaces only its own output files.

In [2]:
TREE_DEPTH = 6
BAGGING_TEMPERATURE = 1.0

INNER_N_SPLITS = 3
MAX_ITERATIONS = 3000
EARLY_STOPPING_ROUNDS = 100
STOPPING_FRACTION = 0.15
THREAD_COUNT = 4

if TREE_DEPTH != 6:
    raise ValueError("Keep TREE_DEPTH = 6 for this bootstrap comparison.")
if not np.isfinite(BAGGING_TEMPERATURE) or BAGGING_TEMPERATURE < 0:
    raise ValueError("BAGGING_TEMPERATURE must be finite and non-negative.")

CATBOOST_PARAMS = {
    "learning_rate": 0.010188851583434083,
    "depth": TREE_DEPTH,
    "l2_leaf_reg": 16.361624028536067,
    "random_strength": 1.4065544206797491,
    "bootstrap_type": "Bayesian",
    "bagging_temperature": BAGGING_TEMPERATURE,
    "boosting_type": "Plain",
    "sampling_frequency": "PerTree",
}

_temperature_tag = format(BAGGING_TEMPERATURE, ".8g").replace(".", "p")
RUN_NAME = f"depth_{TREE_DEPTH}_bayesian_t{_temperature_tag}"
OUTPUT_DIR = RESULTS_DIR / "catboost_bootstrap" / RUN_NAME
MODEL_DIR = OUTPUT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

package_versions = {
    name: version(name)
    for name in ("catboost", "numpy", "pandas", "scikit-learn")
}
print("Depth:", TREE_DEPTH)
print("Bootstrap:", CATBOOST_PARAMS["bootstrap_type"])
print("Bagging temperature:", BAGGING_TEMPERATURE)
print("Random state:", RANDOM_STATE)
print("Output:", OUTPUT_DIR)
print("Versions:", package_versions)

Depth: 6
Bootstrap: Bayesian
Bagging temperature: 1.0
Random state: 42
Output: c:\Users\HP\Documents\GitHub\predict-internet-usage-ivanov-secret\results\catboost_bootstrap\depth_6_bayesian_t1
Versions: {'catboost': '1.2.10', 'numpy': '2.1.3', 'pandas': '2.3.2', 'scikit-learn': '1.9.0'}


## 2. Prepare data

Rebuild Layer A from CSV without changing the shared parquet files. No experimental FGC or task-7 features are added. Numeric NaN stays unchanged; categorical NaN becomes `"missing"`.

In [3]:
raw_train = pd.read_csv(TRAIN_PATH, dtype={ID_COLUMN: str})
raw_test = pd.read_csv(TEST_PATH, dtype={ID_COLUMN: str})
train_features = build_features(raw_train)
test_features = build_features(raw_test)

labeled = train_features.loc[train_features[TARGET].notna()].reset_index(drop=True)
feature_columns = [
    c for c in labeled.columns
    if c not in {ID_COLUMN, TARGET} and not c.startswith("PCIAT-")
]
X = labeled[feature_columns].copy()
y = labeled[TARGET].astype(int)
ids = labeled[ID_COLUMN].astype(str)
X_test = test_features[feature_columns].copy()
test_ids = test_features[ID_COLUMN].astype(str).reset_index(drop=True)

categorical_columns = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()
for frame in (X, X_test):
    for column in categorical_columns:
        values = frame[column].astype("object")
        frame[column] = values.where(values.notna(), "missing").astype(str)
    for column in frame.columns.difference(categorical_columns):
        frame[column] = frame[column].to_numpy(dtype=float, na_value=np.nan)

cv_splits = create_cv_splits(X=X, y=y, ids=ids)
print(f"Train: {X.shape} | Test: {X_test.shape} | Outer folds: {len(cv_splits)}")
print("Class counts:", y.value_counts().sort_index().to_dict())

Train: (2736, 65) | Test: (20, 65) | Outer folds: 5
Class counts: {0: 1594, 1: 730, 2: 378, 3: 34}


## 3. Threshold selection

A multi-start coordinate search maximizes inner-OOF QWK. It is not guaranteed to find the global maximum.

In [4]:
def qwk(y_true, y_pred):
    return float(cohen_kappa_score(
        y_true, y_pred, labels=[0, 1, 2, 3], weights="quadratic"
    ))


def to_classes(predictions, thresholds):
    return np.digitize(np.asarray(predictions), thresholds).astype(int)

In [5]:
def optimize_thresholds(predictions, y, max_passes: int = 12) -> np.ndarray:
    """Fit thresholds to inner-OOF predictions from one outer training fold."""
    p = np.asarray(predictions, dtype=float)
    y = np.asarray(y, dtype=int)
    if p.ndim != 1 or len(p) != len(y) or not np.isfinite(p).all():
        raise ValueError("Predictions and labels must be finite and aligned.")
    order = np.argsort(p, kind="stable")
    ps, ys = p[order], y[order].astype(float)
    n = len(y)
    sy = np.r_[0.0, np.cumsum(ys)]
    sy2 = np.r_[0.0, np.cumsum(ys * ys)]
    unique = np.unique(ps)
    eps = max(1e-8, float(np.ptp(ps)) * 1e-8, float(np.abs(ps).max()) * 1e-10)
    boundaries = np.r_[unique[0] - eps,
                       unique[:-1] + (unique[1:] - unique[:-1]) / 2,
                       unique[-1] + eps]
    classes = np.arange(4)[None, :]
    expected_cost = sy2[-1] - 2 * classes * sy[-1] + classes * classes * n

    def scores(ts):
        indices = np.searchsorted(ps, np.atleast_2d(ts), side="left")
        indices = np.column_stack([np.zeros(len(indices), dtype=int), indices,
                                   np.full(len(indices), n, dtype=int)])
        counts = np.diff(indices, axis=1)
        sums = np.diff(sy[indices], axis=1)
        squares = np.diff(sy2[indices], axis=1)
        observed = (squares - 2 * classes * sums + classes * classes * counts).sum(axis=1)
        expected = (counts * expected_cost).sum(axis=1)
        return 1 - n * observed / expected

    proportions = np.cumsum(np.bincount(y, minlength=4))[:-1] / n
    starts = [np.array([0.5, 1.5, 2.5]), np.quantile(ps, proportions),
              np.quantile(ps, [0.4, 0.75, 0.95]), np.quantile(ps, [0.5, 0.85, 0.99])]
    best_score, best_thresholds = -np.inf, None
    for initial in starts:
        thresholds = np.sort(initial).astype(float)
        for j in range(1, 3):
            if thresholds[j] <= thresholds[j - 1]:
                thresholds[j] = np.nextafter(thresholds[j - 1], np.inf)
        previous = float(scores(thresholds)[0])
        for _ in range(max_passes):
            changed = False
            for j in range(3):
                low = thresholds[j - 1] if j else -np.inf
                high = thresholds[j + 1] if j < 2 else np.inf
                candidates = np.unique(np.r_[boundaries[(boundaries > low) & (boundaries < high)],
                                              thresholds[j]])
                sets = np.repeat(thresholds[None, :], len(candidates), axis=0)
                sets[:, j] = candidates
                values = scores(sets)
                ties = np.flatnonzero(np.isclose(values, values.max(), atol=1e-12, rtol=0))
                chosen = ties[np.argmin(np.abs(candidates[ties] - thresholds[j]))]
                if values[chosen] > previous + 1e-12:
                    thresholds[j] = candidates[chosen]
                    previous = float(values[chosen])
                    changed = True
            if not changed:
                break
        if previous > best_score + 1e-12:
            best_score, best_thresholds = previous, thresholds.copy()
    return best_thresholds

## 4. Inner training

Each inner training fold has a separate stopping holdout. Refit on the full inner training fold, then predict its untouched inner validation fold. Use the median selected tree count for the outer model.

In [6]:
def build_model(iterations):
    return CatBoostRegressor(
        iterations=int(iterations),
        loss_function="RMSE",
        eval_metric="RMSE",
        cat_features=categorical_columns,
        random_seed=RANDOM_STATE,
        thread_count=THREAD_COUNT,
        task_type="CPU",
        allow_writing_files=False,
        verbose=False,
        **CATBOOST_PARAMS,
    )


def fit_inner_oof(X_train, y_train, train_ids, seed):
    positions = np.argsort(train_ids.astype(str).to_numpy())
    inner_cv = StratifiedKFold(
        n_splits=INNER_N_SPLITS, shuffle=True, random_state=seed
    )
    predictions = np.full(len(y_train), np.nan)
    fold_ids = np.zeros(len(y_train), dtype=int)
    tree_counts = []

    for inner_fold, (train_pos, val_pos) in enumerate(
        inner_cv.split(X_train.iloc[positions], y_train.iloc[positions]), start=1
    ):
        train_idx, val_idx = positions[train_pos], positions[val_pos]
        fit_idx, stop_idx = train_test_split(
            train_idx,
            test_size=STOPPING_FRACTION,
            stratify=y_train.iloc[train_idx],
            random_state=seed + inner_fold,
        )
        probe = build_model(MAX_ITERATIONS)
        probe.fit(
            X_train.iloc[fit_idx], y_train.iloc[fit_idx],
            eval_set=(X_train.iloc[stop_idx], y_train.iloc[stop_idx]),
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            use_best_model=True,
        )
        best_iteration = probe.get_best_iteration()
        n_trees = (
            int(best_iteration) + 1
            if best_iteration is not None and best_iteration >= 0
            else int(probe.tree_count_)
        )
        inner_model = build_model(n_trees)
        inner_model.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
        predictions[val_idx] = inner_model.predict(X_train.iloc[val_idx])
        fold_ids[val_idx] = inner_fold
        tree_counts.append(n_trees)

    return predictions, fold_ids, tree_counts

## 5. Cross-validation

The outer splits come from `src.evaluation`, as in the other model notebooks. Each outer fold receives its own training-only thresholds. The test prediction is the mean of the outer models' continuous predictions.

In [ ]:
oof_predictions = np.full(len(y), np.nan)
oof_classes = np.full(len(y), -1, dtype=int)
outer_fold_ids = np.zeros(len(y), dtype=int)
test_predictions_by_fold = np.empty((len(X_test), len(cv_splits)))
fold_records, fold_settings, inner_exports = [], [], []

for fold, (train_idx, val_idx) in enumerate(cv_splits, start=1):
    print(f"Fold {fold}/{len(cv_splits)}: training")
    X_train = X.iloc[train_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    train_ids = ids.iloc[train_idx].reset_index(drop=True)
    inner_seed = int((RANDOM_STATE * 1009 + fold * 9176 + 42) % (2**31 - 1))

    inner_pred, inner_folds, tree_counts = fit_inner_oof(
        X_train, y_train, train_ids, inner_seed
    )
    thresholds = optimize_thresholds(inner_pred, y_train.to_numpy())
    n_trees = max(1, int(np.median(tree_counts)))

    model = build_model(n_trees)
    model.fit(X_train, y_train)
    prediction = model.predict(X.iloc[val_idx])
    classes = to_classes(prediction, thresholds)
    oof_predictions[val_idx] = prediction
    oof_classes[val_idx] = classes
    outer_fold_ids[val_idx] = fold
    test_predictions_by_fold[:, fold - 1] = model.predict(X_test)
    model.save_model(str(MODEL_DIR / f"catboost_fold_{fold}.cbm"))

    fold_records.append({
        "fold": fold,
        "n_trees": n_trees,
        "validation_qwk": qwk(y.iloc[val_idx], classes),
        "macro_f1": f1_score(
            y.iloc[val_idx], classes, labels=[0, 1, 2, 3],
            average="macro", zero_division=0,
        ),
        "recall_class_3": recall_score(
            y.iloc[val_idx], classes, labels=[3], average=None, zero_division=0
        )[0],
    })
    fold_settings.append({
        "fold": fold, "inner_seed": inner_seed, "n_trees": n_trees,
        "inner_tree_counts": tree_counts, "thresholds": thresholds.tolist(),
        "effective_params": model.get_all_params(),
    })
    inner_exports.append(pd.DataFrame({
        ID_COLUMN: train_ids,
        "outer_fold": fold,
        "inner_fold": inner_folds,
        "y_true": y_train,
        "catboost_prediction": inner_pred,
    }))
    print(f"Fold {fold}: QWK={fold_records[-1]['validation_qwk']:.4f}, trees={n_trees}")

fold_results = pd.DataFrame(fold_records)
test_predictions = test_predictions_by_fold.mean(axis=1)

Fold 1/5: training
Fold 1: QWK=0.4708, trees=836
Fold 2/5: training
Fold 2: QWK=0.4858, trees=1298
Fold 3/5: training
Fold 3: QWK=0.4625, trees=705
Fold 4/5: training


## 6. Results

These scores use outer validation predictions with their fold-specific thresholds. No thresholds are fitted or retuned on the pooled outer OOF predictions.

In [ ]:
summary = pd.DataFrame([{
    "model": "CatBoostRegressor",
    "mean_fold_qwk": fold_results["validation_qwk"].mean(),
    "std_fold_qwk": fold_results["validation_qwk"].std(ddof=0),
    "pooled_oof_qwk": qwk(y, oof_classes),
    "macro_f1": f1_score(
        y, oof_classes, labels=[0, 1, 2, 3], average="macro", zero_division=0
    ),
    "recall_class_3": recall_score(
        y, oof_classes, labels=[3], average=None, zero_division=0
    )[0],
}])
confusion = pd.DataFrame(
    confusion_matrix(y, oof_classes, labels=[0, 1, 2, 3]),
    index=[f"true_{c}" for c in range(4)],
    columns=[f"pred_{c}" for c in range(4)],
)
display(fold_results.round(4))
display(summary.round(4))
display(confusion)

effective_columns = [
    "depth", "bootstrap_type", "bagging_temperature", "subsample",
    "sampling_frequency", "boosting_type", "task_type",
]
effective_params = pd.DataFrame([
    {"fold": record["fold"], **{
        key: record["effective_params"].get(key)
        for key in effective_columns
    }}
    for record in fold_settings
])
display(effective_params)

## 7. Save predictions

Use `catboost_prediction`, not `catboost_class`, for blending. Join by `id` and match the outer folds. Test predictions are continuous scores, not a submission.

Select ensemble weights and thresholds inside each outer training fold using matching inner predictions from all models. Do not tune them on pooled outer OOF and report the same score. Correct CatBoost predictions do not fix leakage in other models.

In [ ]:
oof_export = pd.DataFrame({
    ID_COLUMN: ids,
    "fold": outer_fold_ids,
    "y_true": y,
    "catboost_prediction": oof_predictions,
    "catboost_class": oof_classes,
})
test_export = pd.DataFrame({
    ID_COLUMN: test_ids,
    "catboost_prediction": test_predictions,
})
test_fold_export = pd.concat([
    pd.DataFrame({
        ID_COLUMN: test_ids,
        "fold": fold,
        "catboost_prediction": test_predictions_by_fold[:, fold - 1],
    })
    for fold in range(1, len(cv_splits) + 1)
], ignore_index=True)
inner_export = pd.concat(inner_exports, ignore_index=True)

for filename, frame in {
    "catboost_oof.csv": oof_export,
    "catboost_test.csv": test_export,
    "catboost_test_by_fold.csv": test_fold_export,
    "catboost_inner_oof.csv": inner_export,
    "catboost_cv_results.csv": fold_results,
    "catboost_summary.csv": summary,
}.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False)
confusion.to_csv(OUTPUT_DIR / "catboost_confusion.csv")

settings = {
    "model": "CatBoostRegressor",
    "params_source": "Bootstrap experiment based on the depth-6 MVS control",
    "experiment": "catboost_bootstrap",
    "run_name": RUN_NAME,
    "reference_run": "catboost_depth/depth_6",
    "tree_depth": TREE_DEPTH,
    "package_versions": package_versions,
    "python_version": sys.version,
    "params": CATBOOST_PARAMS,
    "random_state": int(RANDOM_STATE),
    "outer_splits": len(cv_splits),
    "inner_splits": INNER_N_SPLITS,
    "max_iterations": MAX_ITERATIONS,
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
    "stopping_fraction": STOPPING_FRACTION,
    "thread_count": THREAD_COUNT,
    "feature_columns": feature_columns,
    "categorical_columns": categorical_columns,
    "categorical_missing_value": "missing",
    "test_prediction": "Mean continuous prediction of outer-fold models",
    "folds": fold_settings,
}
(OUTPUT_DIR / "catboost_settings.json").write_text(
    json.dumps(settings, indent=2), encoding="utf-8"
)
print("Saved to:", OUTPUT_DIR)
display(oof_export.head())
display(test_export.head())
effective_params.to_csv(OUTPUT_DIR / "catboost_effective_params.csv", index=False)